# ARC-AGI Solver — Phase A: RE-ARC Training with Augmentation

Trains the decoder-only transformer on RE-ARC synthetic data (1,000 examples
per task × 400 tasks = 400,000 training pairs) with colour-permutation
augmentation applied per batch.

This is the **Phase A** improvement over the `all_400_arc` baseline, which was
trained on original ARC pairs only (~3–7 examples per task).  Both models use
the same architecture and evaluation pipeline so results are directly comparable.

Checkpoint name: `transformer_crearc_aug_400_best.pt`

**Cell order:** 1 → 2 → 3 → 4 → 5 → 6 → 7

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training on CPU will be extremely slow.')

In [ ]:
# ── Cell 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE         = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR     = f'{DRIVE_BASE}/checkpoints'
DRIVE_TOKENIZED_DIR = f'{DRIVE_BASE}/tokenized'

Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(DRIVE_TOKENIZED_DIR).mkdir(parents=True, exist_ok=True)

print(f'Checkpoint dir:  {DRIVE_CKPT_DIR}')
print(f'Tokenized dir:   {DRIVE_TOKENIZED_DIR}')

In [ ]:
# ── Cell 3: Clone repo and download ARC training data ───────────────────────
import os, io, subprocess, urllib.request, zipfile
from pathlib import Path

REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
REPO_DIR    = f'/content/{REPO}'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git', REPO_DIR],
                   check=True)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

log = subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-3'],
                     capture_output=True, text=True)
print(log.stdout)

# Download ARC training data
DATA_DIR  = Path('data')
train_dir = DATA_DIR / 'training'
if not train_dir.exists() or len(list(train_dir.glob('*.json'))) < 400:
    print('Downloading ARC training data...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        dest = DATA_DIR / 'training'
        dest.mkdir(exist_ok=True)
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list(train_dir.glob('*.json')))} tasks")
else:
    print(f"ARC training data already present ({len(list(train_dir.glob('*.json')))} tasks)")

In [ ]:
# ── Cell 4: Download RE-ARC synthetic data ──────────────────────────────────
# 400 tasks × 1,000 examples each → data/re_arc/  (~110 MB zip)
import subprocess, sys
from pathlib import Path

RE_ARC_DIR = Path('data/re_arc')
if RE_ARC_DIR.exists() and len(list(RE_ARC_DIR.glob('*.json'))) >= 400:
    print(f'RE-ARC already present ({len(list(RE_ARC_DIR.glob("*.json")))} tasks)')
else:
    print('Downloading RE-ARC synthetic data...')
    result = subprocess.run([sys.executable, 'scripts/download_re_arc.py'],
                            capture_output=True, text=True)
    out = result.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    print(f'RE-ARC: {len(list(RE_ARC_DIR.glob("*.json")))} tasks')

In [ ]:
# ── Cell 5: Configuration ────────────────────────────────────────────────────
# Phase A: RE-ARC data (1,000 synthetic examples per task) with colour-
# permutation augmentation applied during training.
# Compare against all_400_arc (original ARC pairs only, ~3–7 per task).

RUN_NAME        = 'rearc_aug_400'   # checkpoint: transformer_crearc_aug_400_best.pt
K_CONTEXT       = 3
D_MODEL         = 512
N_HEADS         = 8
N_LAYERS        = 6
LR              = 3e-4
WARMUP_EPOCHS   = 5
EPOCHS          = 300
STEPS_PER_EPOCH = 400
MAX_TOKENS      = 96000
SAVE_EVERY      = 25
LOG_EVERY       = 1
LOG_FILE        = f'results/train_{RUN_NAME}.log'

# All 400 ARC training tasks
TASK_IDS = [
            '007bbfb7', '00d62c1b', '017c7c7b', '025d127b', '045e512c',
            '0520fde7', '05269061', '05f2a901', '06df4c85', '08ed6ac7',
            '09629e4f', '0962bcdd', '0a938d79', '0b148d64', '0ca9ddb6',
            '0d3d703e', '0dfd9992', '0e206a2e', '10fcaaa3', '11852cab',
            '1190e5a7', '137eaa0f', '150deff5', '178fcbfb', '1a07d186',
            '1b2d62fb', '1b60fb0c', '1bfc4729', '1c786137', '1caeab9d',
            '1cf80156', '1e0a9b12', '1e32b0e9', '1f0c79e5', '1f642eb9',
            '1f85a75f', '1f876c06', '1fad071e', '2013d3e2', '2204b7a8',
            '22168020', '22233c11', '2281f1f4', '228f6490', '22eb0ac0',
            '234bbc79', '23581191', '239be575', '23b5c85d', '253bf280',
            '25d487eb', '25d8a9c8', '25ff71a9', '264363fd', '272f95fa',
            '27a28665', '28bf18c6', '28e73c20', '29623171', '29c11459',
            '29ec7d0e', '2bcee788', '2bee17df', '2c608aff', '2dc579da',
            '2dd70a9a', '2dee498d', '31aa019c', '321b1fc6', '32597951',
            '3345333e', '3428a4f5', '3618c87e', '3631a71a', '363442ee',
            '36d67576', '36fdfd69', '3906de3d', '39a8645d', '39e1d7f9',
            '3aa6fb7a', '3ac3eb23', '3af2c5a8', '3bd67248', '3bdb4ada',
            '3befdf3e', '3c9b0459', '3de23699', '3e980e27', '3eda0437',
            '3f7978a0', '40853293', '4093f84a', '41e4d17e', '4258a5f9',
            '4290ef0e', '42a50994', '4347f46a', '444801d8', '445eab21',
            '447fd412', '44d8ac46', '44f52bb0', '4522001f', '4612dd53',
            '46442a0e', '469497ad', '46f33fce', '47c1f68c', '484b58aa',
            '48d8fb45', '4938f0c2', '496994bd', '49d1d64f', '4be741c5',
            '4c4377d9', '4c5c2cf0', '50846271', '508bd3b6', '50cb2852',
            '5117e062', '5168d44c', '539a4f51', '53b68214', '543a7ed5',
            '54d82841', '54d9e175', '5521c0d9', '5582e5ca', '5614dbcf',
            '56dc2b01', '56ff96f3', '57aa92db', '5ad4f10b', '5bd6f4ac',
            '5c0a986e', '5c2c9af4', '5daaa586', '60b61512', '6150a2bd',
            '623ea044', '62c24649', '63613498', '6430c8c4', '6455b5f5',
            '662c240a', '67385a82', '673ef223', '6773b310', '67a3c6ac',
            '67a423a3', '67e8384a', '681b3aeb', '6855a6e4', '68b16354',
            '694f12f3', '6a1e5592', '6aa20dc0', '6b9890af', '6c434453',
            '6cdd2623', '6cf79266', '6d0160f0', '6d0aefbc', '6d58a25d',
            '6d75e8bb', '6e02f1e3', '6e19193c', '6e82a1ae', '6ecd11f4',
            '6f8cd79b', '6fa7a44f', '72322fa7', '72ca375d', '73251a56',
            '7447852a', '7468f01a', '746b3537', '74dd1130', '75b8110e',
            '760b3cac', '776ffc46', '77fdfe62', '780d0b14', '7837ac64',
            '794b24be', '7b6016b9', '7b7f7511', '7c008303', '7ddcd7ec',
            '7df24a62', '7e0986d6', '7f4411dc', '7fe24cdd', '80af3007',
            '810b9b61', '82819916', '83302e8f', '834ec97d', '8403a5d5',
            '846bdb03', '855e0971', '85c4e7cd', '868de0fa', '8731374e',
            '88a10436', '88a62173', '890034e9', '8a004b2b', '8be77c9e',
            '8d5021e8', '8d510a79', '8e1813be', '8e5a5113', '8eb1be9a',
            '8efcae92', '8f2ea7aa', '90c28cc7', '90f3ed37', '913fb3ed',
            '91413438', '91714a58', '9172f3a0', '928ad970', '93b581b8',
            '941d9a10', '94f9d214', '952a094c', '9565186b', '95990924',
            '963e52fc', '97999447', '97a05b5b', '98cf29f8', '995c5fa3',
            '99b1bc43', '99fa7670', '9aec4887', '9af7a82c', '9d9215db',
            '9dfd6313', '9ecd008a', '9edfc990', '9f236235', 'a1570a43',
            'a2fd1cf0', 'a3325580', 'a3df8b1e', 'a416b8f3', 'a48eeaf7',
            'a5313dff', 'a5f85a15', 'a61ba2ce', 'a61f2674', 'a64e4611',
            'a65b410d', 'a68b268e', 'a699fb00', 'a740d043', 'a78176bb',
            'a79310a0', 'a85d4709', 'a87f7484', 'a8c38be5', 'a8d7556c',
            'a9f96cdd', 'aabf363d', 'aba27056', 'ac0a08a4', 'ae3edfdc',
            'ae4f1146', 'aedd82e4', 'af902bf9', 'b0c4d837', 'b190f7f5',
            'b1948b0a', 'b230c067', 'b27ca6d3', 'b2862040', 'b527c5c6',
            'b548a754', 'b60334d2', 'b6afb2da', 'b7249182', 'b775ac94',
            'b782dc8a', 'b8825c91', 'b8cdaf2b', 'b91ae062', 'b94a9452',
            'b9b7f026', 'ba26e723', 'ba97ae07', 'bb43febb', 'bbc9ae5d',
            'bc1d5164', 'bd4472b8', 'bda2d7a6', 'bdad9b1f', 'be94b721',
            'beb8660c', 'c0f76784', 'c1d99e64', 'c3e719e8', 'c3f564a4',
            'c444b776', 'c59eb873', 'c8cbb738', 'c8f0f002', 'c909285e',
            'c9e6f938', 'c9f8e694', 'caa06a1f', 'cbded52d', 'cce03e0d',
            'cdecee7f', 'ce22a75a', 'ce4f8723', 'ce602527', 'ce9e57f2',
            'cf98881b', 'd037b0a7', 'd06dbe63', 'd07ae81c', 'd0f5fe59',
            'd10ecb37', 'd13f3404', 'd22278a0', 'd23f8c26', 'd2abd087',
            'd364b489', 'd406998b', 'd43fd935', 'd4469b4b', 'd4a91cb9',
            'd4f3cd78', 'd511f180', 'd5d6de2d', 'd631b094', 'd687bc17',
            'd6ad076f', 'd89b689b', 'd8c310e9', 'd90796e8', 'd9f24cd1',
            'd9fac9be', 'dae9d2b5', 'db3e9e38', 'db93a21d', 'dbc1a6ce',
            'dc0a314f', 'dc1df850', 'dc433765', 'ddf7fa4f', 'de1cd16c',
            'ded97339', 'e179c5f4', 'e21d9049', 'e26a3af2', 'e3497940',
            'e40b9e2f', 'e48d4e1a', 'e5062a87', 'e509e548', 'e50d258f',
            'e6721834', 'e73095fd', 'e76a88a6', 'e8593010', 'e8dc4411',
            'e9614598', 'e98196ab', 'e9afcf9a', 'ea32f347', 'ea786f4a',
            'eb281b96', 'eb5a1d5d', 'ec883f72', 'ecdecbb3', 'ed36ccf7',
            'ef135b50', 'f15e1fac', 'f1cefba8', 'f25fbde4', 'f25ffba3',
            'f2829549', 'f35d900a', 'f5b8619d', 'f76d97a5', 'f8a8fe49',
            'f8b3ba0a', 'f8c80d96', 'f8ff0b80', 'f9012d9b', 'fafffa47',
            'fcb5c309', 'fcc82909', 'feca6190', 'ff28f65a', 'ff805c23',
        ]

# 40 held-out tasks used for checkpoint selection (ARC leave-one-out exact match)
VAL_TASK_IDS = [
            '0a938d79', '0b148d64', '0d3d703e', '0dfd9992', '178fcbfb',
            '1bfc4729', '1e0a9b12', '1f876c06', '2013d3e2', '23b5c85d',
            '27a28665', '447fd412', '4522001f', '54d82841', '57aa92db',
            '5bd6f4ac', '623ea044', '6455b5f5', '67a3c6ac', '67a423a3',
            '681b3aeb', '6855a6e4', '6aa20dc0', '6d58a25d', '6d75e8bb',
            '73251a56', '74dd1130', '7b7f7511', '85c4e7cd', '8d5021e8',
            '963e52fc', 'a740d043', 'bc1d5164', 'cf98881b', 'd364b489',
            'd43fd935', 'd4a91cb9', 'd89b689b', 'ddf7fa4f', 'eb281b96',
        ]

print(f'Run:         {RUN_NAME}')
print(f'Train tasks: {len(TASK_IDS)} (all 400 ARC training tasks)')
print(f'Val tasks:   {len(VAL_TASK_IDS)} (ARC LOO exact match — checkpoint criterion)')
print(f'Epochs:      {EPOCHS} × {STEPS_PER_EPOCH} steps = {EPOCHS*STEPS_PER_EPOCH:,} total')
print(f'd_model:     {D_MODEL},  heads: {N_HEADS},  layers: {N_LAYERS},  k_ctx: {K_CONTEXT}')
print(f'LR:          {LR},  warmup: {WARMUP_EPOCHS} epochs,  max_tokens: {MAX_TOKENS}')

In [ ]:
# ── Cell 6: Pre-tokenise RE-ARC examples ────────────────────────────────────
# Encodes every RE-ARC pair once → data/tokenized/<task_id>.npz
# Synced to/from Drive so it only runs once across sessions.
import subprocess, sys, shutil
from pathlib import Path

local_tok = Path('data/tokenized')
drive_tok = Path(DRIVE_TOKENIZED_DIR)
local_tok.mkdir(parents=True, exist_ok=True)

# Restore any files already on Drive
restored = 0
for p in drive_tok.glob('*.npz'):
    dest = local_tok / p.name
    if not dest.exists():
        shutil.copy2(p, dest)
        restored += 1
if restored:
    print(f'Restored {restored} .npz file(s) from Drive.')

# Tokenise anything still missing
already  = {p.stem for p in local_tok.glob('*.npz')}
missing  = [tid for tid in TASK_IDS if tid not in already]

if not missing:
    print(f'All {len(TASK_IDS)} tasks already tokenised.')
else:
    print(f'Tokenising {len(missing)} tasks...')
    result = subprocess.run(
        [sys.executable, 'scripts/pretokenize.py', '--tasks'] + missing,
        capture_output=True, text=True,
    )
    out = result.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])

# Sync new files to Drive
synced = 0
for p in local_tok.glob('*.npz'):
    dest = drive_tok / p.name
    if not dest.exists():
        shutil.copy2(p, dest)
        synced += 1
if synced:
    print(f'Saved {synced} new .npz file(s) to Drive.')
print(f'Tokenized files on disk: {len(list(local_tok.glob("*.npz")))}')

In [ ]:
# ── Cell 7: Run training ─────────────────────────────────────────────────────
# Checkpoint saved to Drive: transformer_crearc_aug_400_best.pt
# Criterion: highest ARC LOO exact match on VAL_TASK_IDS.
import subprocess, sys

cmd = [
    sys.executable, '-u', 'scripts/train_transformer.py',
    '--data-source',     'rearc',          # RE-ARC synthetic data (1000/task)
    '--task-ids',        *TASK_IDS,
    '--val-arc-task-ids', *VAL_TASK_IDS,
    '--run-name',        RUN_NAME,
    '--epochs',          str(EPOCHS),
    '--steps-per-epoch', str(STEPS_PER_EPOCH),
    '--k-context',       str(K_CONTEXT),
    '--d-model',         str(D_MODEL),
    '--n-heads',         str(N_HEADS),
    '--n-layers',        str(N_LAYERS),
    '--lr',              str(LR),
    '--warmup-epochs',   str(WARMUP_EPOCHS),
    '--max-tokens',      str(MAX_TOKENS),
    '--max-seq-len',     '6000',
    '--save-every',      str(SAVE_EVERY),
    '--log-every',       str(LOG_EVERY),
    '--log',             LOG_FILE,
    '--ckpt-dir',        DRIVE_CKPT_DIR,
]

print(f'Checkpoints → {DRIVE_CKPT_DIR}')
print(f'Log         → {LOG_FILE}')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 8: View training log (last 40 lines) ───────────────────────────────
from pathlib import Path
log = Path(LOG_FILE)
if log.exists():
    lines = log.read_text().splitlines()
    print(f'Log: {log}  ({len(lines)} lines)')
    print('\n'.join(lines[-40:]))
else:
    print('Log file not found — training may not have started yet.')